# the prototype code for the 2025-summer-project

### the code will be composed of following parts:

1. datasets: get dataset and generate dataloaders for cifar10,etc
2. samplers: generating samples of standard guassian noise/ corrupted image/ lognorm times
3. paths: the conceptual/ideal path for the image transformation process
4. models&vectorfields: neural networks with Unet Architecture / other pretrained model as backbone
5. trainer: define the loss function and deal with all data generated in the training process
6. simulators: eulars and heun simulators for image generation|restoration
7. utils: codes for utils functions (visualization + model_utils)  
7. data: the raw data for training
8. checkpoints: save the models params and other key information
9. results : save the image outputs


### task1:learn and modify the ref-project from ZHB.

- [x] some minor format improvements and additional comments
- [x] change the get_dataloader functions in the datasets package
- [x] change the trainer function: add param init_channel.
> - modify the save_checkpoints function to dump the model in checkpoints
> - modify the visualization function   
> - modify the path class: use noise(Sampler) and data(DataLoader) sampling a x_t point
> try to understand what's the methods(schduler and early stop...) are used in the **trainer** function   

----



### task2: from flow-matching to meanflow

----

## Datasets

In [174]:
%%writefile datasets/__init__.py

##datasets init
from .mnist import get_mnist_dataloader
from .cifar10 import get_cifar10_dataloader

# define the interface
__all__ = ["get_mnist_dataloader","get_cifar10_dataloader"]

Overwriting datasets/__init__.py


In [175]:
%%writefile datasets/cifar10.py
# get the dataloader with a transformed dataset.

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from typing import Tuple, Optional
def get_cifar10_dataloader(
        root: str = "data/",
        batch_size: int = 128,
        num_workers: int = 4,
        download: bool = True
) -> DataLoader:
    
    """the function to get a dataloader"""
    mean = (0.5, 0.5, 0.5)
    std = (0.5, 0.5, 0.5)

    
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    train_dataset = datasets.CIFAR10(
        root = root,
        train = True,
        download = download,
        transform = train_transform
    )

    train_dataloader = DataLoader(
        train_dataset,
        batch_size = batch_size,
        shuffle = True,
        num_workers = num_workers,)
    
    return train_dataloader



Overwriting datasets/cifar10.py


In [176]:
%%writefile datasets/mnist.py
# get the dataloader with a transformed dataset.

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from typing import Tuple, Optional
def get_mnist_dataloader(
        root: str = "data/",
        batch_size: int = 128,
        num_workers: int = 4,
        download: bool = True
) -> DataLoader:
    
    """the function to get a dataloader"""
    mean = (0.5)
    std = (0.5)

    
    train_transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])

    train_dataset = datasets.MNIST(
        root = root,
        train = True,
        download = download,
        transform = train_transform
    )

    train_dataloader = DataLoader(
        train_dataset,
        batch_size = batch_size,
        shuffle = True,
        num_workers = num_workers,)
    
    return train_dataloader
#batch_size, c, H, W


Overwriting datasets/mnist.py


## Samplers

In [177]:
%%writefile samplers/__init__.py
from .base import Sampler
from .gaussian import IsotropicGaussian

__all__ = ["Sampler","IsotropicGaussian"]

Overwriting samplers/__init__.py


In [178]:
%%writefile samplers/base.py

#base class for sampling time.
from abc import ABC, abstractmethod
from typing import Tuple
import torch
import torch.nn as nn

class Sampler(nn.Module, ABC):
    """interface to sample from p_init"""
    
    def __init__(self, sample_shape: Tuple[int]):
        """
        Args:
            sample_shape (Tuple[int]): (c, h, w)
        """
        super().__init__()
        self.sample_shape = sample_shape
        self.register_buffer("dummy", torch.tensor(0,))
        
    def shape(self) -> Tuple[int]:
        return self.sample_shape

    @abstractmethod
    def sample(self, num_samples: int) -> torch.Tensor:
        # return num_samples from p_init, (num_samples, shape)
        pass

Overwriting samplers/base.py


In [179]:
%%writefile samplers/gaussian.py
# generate guassian noise 
 
# define base class
from .base import Sampler
from abc import ABC, abstractmethod
from typing import Tuple
import torch
import torch.nn as nn


class IsotropicGaussian(Sampler):
    def __init__(self, sample_shape: Tuple[int], std: float = 1.0):
        super().__init__(sample_shape)
        self.register_buffer("std", torch.tensor(std))

    def sample(self, num_samples: int) -> torch.Tensor:
        return self.std * torch.randn(num_samples, *self.sample_shape, device=self.std.device)


Overwriting samplers/gaussian.py


## Paths

In [180]:
%%writefile paths/__init__.py
##paths:
# using the data from datasets + samplers return the velocity field as Reference.

from .base import ConditionalPath
from .gaussian import GaussianConditionalPath
from .linear_gaussian import LinearAlpha, LinearBeta

# import abstract and specific objects for use
__all__ = ["ConditionalPath", "GaussianConditionalPath", 
           "LinearAlpha", "LinearBeta"]

Overwriting paths/__init__.py


In [181]:

%%writefile paths/base.py
from abc import ABC, abstractmethod
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader
from samplers import Sampler



class ConditionalPath(nn.Module, ABC):
    def __init__(self, noise:Sampler):
        super().__init__()
        self.noise = noise 
        # the value of selected data/noise doesn't need to be shared with other class
        self.register_buffer("dummy", torch.tensor(0, ))
    
    @abstractmethod
    def sample_conditional_path(self, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Samples from the conditional distribution p_t(x|z), for neural network params
        
        Args:
            - z: conditioning variable (num_samples, c, h, w)
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - xt: samples from p_t(x|z), (num_samples, c, h, w)
        """
        pass

    @abstractmethod
    def conditional_velocity(self, xt: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the conditional vector field u_t(x|z) as a reference
        
        Args:
            - x: position variable (num_samples, c, h, w)
            - z: conditioning variable (num_samples, c, h, w)
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - conditional_vector_field: conditional vector field (num_samples, c, h, w)
        """
        pass


      

Overwriting paths/base.py


In [182]:
%%writefile paths/gaussian.py
# metadata for guassian path
from abc import ABC, abstractmethod
from .base import ConditionalPath
import torch
from torch.func import jacrev, vmap
from typing import Tuple
from samplers import IsotropicGaussian
from torch.utils.data import DataLoader

class Alpha(ABC):
   
    
    
    @abstractmethod
    def __call__(self, t:torch.Tensor)-> torch.Tensor:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 0.0, self(1.0) = 1.0.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - alpha_t (num_samples, 1, 1, 1)
        """
        pass
       
    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - d/dt alpha_t (num_samples, 1, 1, 1)
        """
        t = t.unsqueeze(1)
        da_dt = vmap(jacrev(self))(t)
        return da_dt.view(-1, 1, 1, 1).to(t.device)
    
class Beta(ABC):
        
    
    @abstractmethod
    def __call__(self, t:torch.Tensor)-> torch.Tensor:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 0.0, self(1.0) = 1.0.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - alpha_t (num_samples, 1, 1, 1)
        """
        pass
       
    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - d/dt alpha_t (num_samples, 1, 1, 1)
        """
        t = t.unsqueeze(1)
        db_dt = vmap(jacrev(self))(t)
        return db_dt.view(-1, 1, 1, 1).to(t.device)

class GaussianConditionalPath(ConditionalPath):
    def __init__(self, p_init_shape: Tuple[int], alpha: Alpha, beta: Beta):
        noise = IsotropicGaussian(p_init_shape)
        super().__init__(noise)
        self.alpha = alpha
        self.beta = beta

    def sample_conditional_path(self, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        x_init = self.noise.sample(z.shape[0])
        xt = self.alpha(t) * z + self.beta(t) * x_init
        return xt

    def conditional_velocity(self, xt: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        alpha_t = self.alpha(t)
        beta_t = self.beta(t)
        d_alpha_dt = self.alpha.dt(t)
        d_beta_dt = self.beta.dt(t)
        return (d_alpha_dt - d_beta_dt * alpha_t / beta_t) * z + d_beta_dt / beta_t * xt


Overwriting paths/gaussian.py


In [183]:
%%writefile paths/linear_gaussian.py
from .gaussian import Alpha, Beta
import torch

class LinearAlpha(Alpha):
    def __init__(self):
        super().__init__()

    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        return t

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        return torch.ones_like(t)

class LinearBeta(Beta):
    def __init__(self):
        super().__init__()

    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        return 1 - t

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        return - torch.ones_like(t)


Overwriting paths/linear_gaussian.py


## Models

In [184]:
%%writefile models/__init__.py
## the model: predict the average vectorfield formed by guassian paths

# the model is backboned with unet, which is composed of Encoder, decoder, midcoder and skip connection.
# residual layers are emphasized and fourier encoder is adopted for time perception


from .fm_unet import UNetVelocity
from .base import NNVelocity

__all__ = ["NNVelocity","UNetVelocity"]



Overwriting models/__init__.py


In [185]:
%%writefile models/fm_unet/fourier_encoder.py
#Fourier Encoder for better time perception

import torch
import torch.nn as nn
import math

class FourierEncoder(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        assert (dim % 2 == 0)
        self.half_dim = dim // 2
        self.freqs = nn.Parameter(torch.randn(1, self.half_dim)) # (1, half_dim)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - t: (bs, 1, 1, 1)
        Returns:
        - embeddings: (bs, dim)
        """
        t = t.view(-1, 1) # (bs, 1)
        theta = t * self.freqs * 2 * math.pi # (bs, half_dim)
        sin_embed = torch.sin(theta) # (bs, half_dim)
        cos_embed = torch.cos(theta) # (bs, half_dim)
        return torch.cat([sin_embed, cos_embed], dim=-1) * math.sqrt(2) # (bs, dim)


Overwriting models/fm_unet/fourier_encoder.py


In [186]:

%%writefile models/fm_unet/residual_layer.py
# core unit of the unet: residual layer
import torch
import torch.nn as nn
from .fourier_encoder import FourierEncoder

class ResidualLayer(nn.Module):
    def __init__(self, channels: int, time_embed_dim: int, y_embed_dim: int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.SiLU(),
            nn.BatchNorm2d(channels),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        )
        self.block2 = nn.Sequential(
            nn.SiLU(),
            nn.BatchNorm2d(channels),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        )
        # Converts (bs, time_embed_dim) -> (bs, channels)
        self.time_adapter = nn.Sequential(
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, channels)
        )
        # Converts (bs, y_embed_dim) -> (bs, channels)
        self.y_adapter = nn.Sequential(
            nn.Linear(y_embed_dim, y_embed_dim),
            nn.SiLU(),
            nn.Linear(y_embed_dim, channels)
        )

    def forward(self, x: torch.Tensor, t_embed: torch.Tensor, y_embed: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - x: (bs, c, h, w)
        - t_embed: (bs, t_embed_dim)
        - y_embed: (bs, y_embed_dim)
        """
        res = x.clone() # (bs, c, h, w)
        x = self.block1(x) # (bs, c, h, w)

        t_embed = self.time_adapter(t_embed).unsqueeze(-1).unsqueeze(-1) # (bs, c, 1, 1)
        x = x + t_embed
        y_embed = self.y_adapter(y_embed).unsqueeze(-1).unsqueeze(-1) # (bs, c, 1, 1)
        x = x + y_embed

        x = self.block2(x) # (bs, c, h, w)
        x = x + res # (bs, c, h, w)

        return x


Overwriting models/fm_unet/residual_layer.py


In [187]:
%%writefile models/fm_unet/modules.py
# modules of unet: encoder, midcoder and decoder
import torch
import torch.nn as nn
from .residual_layer import ResidualLayer

class Encoder(nn.Module):
    def __init__(self, channels_in: int, channels_out: int, num_residual_layers: int, t_embed_dim: int, y_embed_dim: int):
        super().__init__()
        self.res_blocks = nn.ModuleList([
            ResidualLayer(channels_in, t_embed_dim, y_embed_dim) for _ in range(num_residual_layers)
        ])
        self.downsample = nn.Sequential(
            nn.Conv2d(in_channels = channels_in,
                    out_channels= channels_out,
                    kernel_size= 3, 
                    stride=2,
                    padding = 1),
            nn.BatchNorm2d(channels_out),
            nn.SiLU()
        )

    def forward(self, x: torch.Tensor, t_embed: torch.Tensor, y_embed: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - x: (bs, c_in, h, w)
        - t_embed: (bs, t_embed_dim)
        - y_embed: (bs, y_embed_dim)
        """
        # Pass through residual blocks: (bs, c_in, h, w) -> (bs, c_in, h, w)
        for block in self.res_blocks:
            x = block(x, t_embed, y_embed)

        # Downsample: (bs, c_in, h, w) -> (bs, c_out, h // 2, w // 2)
        x = self.downsample(x)

        return x

class Midcoder(nn.Module):
    def __init__(self, channels: int, num_residual_layers: int, t_embed_dim: int, y_embed_dim: int):
        super().__init__()
        self.res_blocks = nn.ModuleList([
            ResidualLayer(channels, t_embed_dim, y_embed_dim) for _ in range(num_residual_layers)
        ])

    def forward(self, x: torch.Tensor, t_embed: torch.Tensor, y_embed: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - x: (bs, c, h, w)
        - t_embed: (bs, t_embed_dim)
        - y_embed: (bs, y_embed_dim)
        """
        # Pass through residual blocks: (bs, c, h, w) -> (bs, c, h, w)
        for block in self.res_blocks:
            x = block(x, t_embed, y_embed)

        return x

class Decoder(nn.Module):
    def __init__(self, channels_in: int, channels_out: int, num_residual_layers: int, t_embed_dim: int, y_embed_dim: int):
        super().__init__()
        self.upsample = nn.Sequential(
            nn.Conv2d(channels_in, channels_in//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels_in//2),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode='bilinear'),
            nn.Conv2d(channels_in//2, channels_out, kernel_size=3, padding=1),
        )      
        # here, given the concatenation of x_res and x, the number of channels doubles.
        # hence, the upsample need a extra conv + batchnorm to decrease the channels.
        self.res_blocks = nn.ModuleList([
            ResidualLayer(channels_out, t_embed_dim, y_embed_dim) for _ in range(num_residual_layers)
        ])

    def forward(self, x: torch.Tensor, t_embed: torch.Tensor, y_embed: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - x: (bs, c, h, w)
        - t_embed: (bs, t_embed_dim)
        - y_embed: (bs, y_embed_dim)
        """
        # Upsample: (bs, c_in, h, w) -> (bs, c_out, 2 * h, 2 * w)
        x = self.upsample(x)

        # Pass through residual blocks: (bs, c_out, h, w) -> (bs, c_out, 2 * h, 2 * w)
        for block in self.res_blocks:
            x = block(x, t_embed, y_embed)

        return x


Overwriting models/fm_unet/modules.py


In [199]:
%%writefile models/fm_unet/unet.py
#composition of unet architecture
import torch
import torch.nn as nn
from ..base import NNVelocity
from .fourier_encoder import FourierEncoder
from .modules import Encoder, Midcoder, Decoder
from typing import List


class UNetVelocity(NNVelocity):
    def __init__(self, init_channel, channels: List[int], num_residual_layers: int, t_embed_dim: int, y_embed_dim: int, num_categories: int):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(init_channel, channels[0], kernel_size=3, padding=1),
            nn.BatchNorm2d(channels[0]),
            nn.SiLU()
        )
        self.time_embedder = FourierEncoder(t_embed_dim)
        self.y_embedder = nn.Embedding(num_categories+1, y_embed_dim)

        encoders = []
        decoders = []
        for (curr_channel, next_channel) in zip(channels[:-1], channels[1:]):
            encoders.append(Encoder(curr_channel, next_channel, num_residual_layers, t_embed_dim, y_embed_dim))
            decoders.append(Decoder(2*next_channel, curr_channel, num_residual_layers, t_embed_dim, y_embed_dim))
        self.encoders = nn.ModuleList(encoders)
        self.decoders = nn.ModuleList(reversed(decoders))
        self.midcoder = Midcoder(channels[-1], num_residual_layers, t_embed_dim, y_embed_dim)

        self.final_conv = nn.Conv2d(channels[0],init_channel, kernel_size=3, padding=1)


    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - x: (bs, c, h, w)
        - t: (bs, 1, 1, 1)
        - y: (bs,)
        Returns:
        - u_t^theta(x|y): (bs, c, h, w)
        """
        # Embed t and y
        res = []
        t_embed = self.time_embedder(t) # (bs, t_embed_dim)
        y_embed = self.y_embedder(y) # (bs, y_embed_dim)
        
        x = self.init_conv(x) # (bs, c[1], h, w)
        
        for encoder in self.encoders:
            x = encoder(x, t_embed, y_embed) # (bs, c[i], h, w) -> (bs, c[i+1], h//2, w//2)
            res.append(x.clone()) # push res into the stack

        x = self.midcoder(x, t_embed, y_embed)

        for decoder in self.decoders:
            x_res = res.pop() # (bs, c[i], h, w) last-in, first-out
            # do concatenation , double the channels
            x = torch.cat([x, x_res], dim=1) # (bs, c[i]*2, h, w)
            x = decoder(x, t_embed, y_embed) # (bs, c[i-1], 2*h, 2*w)

        x = self.final_conv(x)

        return x


Overwriting models/fm_unet/unet.py


## vectorfields and trainer

In [189]:
%%writefile vector_fields/base.py 
from abc import ABC, abstractmethod
import torch

class VectorField(ABC):
    @abstractmethod
    def velocity(self, xt: torch.Tensor, t: torch.Tensor, **kwargs) -> torch.Tensor:
        """
        Returns the drift coefficient of the ODE.
        Args:
            - xt: state at time t, shape (bs, c, h, w)
            - t: time, shape (bs, 1)
        Returns:
            - velocity: (bs, c, h, w)
        """
        pass


Overwriting vector_fields/base.py


In [190]:
%%writefile vector_fields/cfg_vector_field.py
from . import VectorField
from models import NNVelocity
import torch
import torch.nn as nn
from abc import ABC, abstractmethod


class CFGVectorField(VectorField):
    def __init__(self, model: NNVelocity, guidance_scale: float = 1.0):
        super().__init__()
        self.model = model
        self.guidance_scale = guidance_scale

    def velocity(self, xt: torch.Tensor, t: torch.Tensor, y: torch.Tensor, categories: int = 10) -> torch.Tensor:
        guided_velocity = self.model(xt, t, y)
        unguided_y = torch.ones_like(y) * categories
        unguided_velocity = self.model(xt, t, unguided_y)
        w = self.guidance_scale
        return w * guided_velocity + (1 - w) * unguided_velocity


Overwriting vector_fields/cfg_vector_field.py


In [191]:
%%writefile trainers/base.py
import torch
import torch.nn as nn
from abc import ABC, abstractmethod
from tqdm import tqdm
from utils import model_size_mib, save_checkpoint, draw_loss_curve

class Trainer(ABC):
    def __init__(self, model: nn.Module):
        self.model = model

    @abstractmethod
    def get_train_loss(self, data, device) -> torch.Tensor:
        pass

    def get_optimizer(self, lr: float):
        return torch.optim.Adam(self.model.parameters(), lr=lr)

    def get_scaler(self):
        device_name = 'cuda' if torch.cuda.is_available else 'cpu'
        return torch.amp.GradScaler(device_name)

    def get_grad_clip_norm(self) -> float:
        return 1.0

    def get_scheduler(self, optimizer: torch.optim.Optimizer):
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=optimizer,
            mode="min",
            factor=0.5,
            patience=1000,
            min_lr=1e-6
        )

    def train(self, num_epochs: int, device: torch.device,
              dataloader: torch.utils.data.DataLoader, project_name: str, lr: float=1e-3):
        # print the model size
        model_size = model_size_mib(self.model)
        print(f'Training model with size: {model_size} MiB')

        # train start
        self.model.to(device)
        self.model.train()

        #trainer modules setup
        optimizer = self.get_optimizer(lr)
        scheduler = self.get_scheduler(optimizer)
        scaler = self.get_scaler()
        clip_norm = self.get_grad_clip_norm()

        loss_history = []
        pbar = tqdm(enumerate(range(num_epochs)))

        # train loop
        for index, epoch in pbar:
            for batch_idx, data in enumerate(dataloader):
                optimizer.zero_grad()
                with torch.amp.autocast(device_type=device.type):
                    loss = self.get_train_loss(data=data, device=device)

                # Mixed precision backward + gradient clip
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=clip_norm)
                scaler.step(optimizer)
                scaler.update()

                scheduler.step(loss.item())  # default: ReduceLROnPlateau
                loss_history.append(loss.item())
                pbar.set_description(f"Epoch {index} batch {batch_idx}, Loss: {loss.item():.4f}")

        # train finish
        self.model.eval()
        draw_loss_curve(loss_history, num_epochs=num_epochs,
                        batch_size=dataloader.batch_size, project_name=project_name)
        save_checkpoint(self.model, optimizer, scheduler, scaler,
                        epoch=num_epochs, loss_history=loss_history,
                        project_name=project_name, filename=f"{project_name}_model.pth")


Overwriting trainers/base.py


In [ ]:
%%writefile trainers/fm_trainer.py
import torch
import torch.nn as nn
from .base import Trainer
from paths import ConditionalPath
from models import NNVelocity

class CFGTrainerFM(Trainer):
    def __init__(self, path: ConditionalPath, model: NNVelocity, eta: float):
        assert (eta <= 1) and (eta >= 0)
        super().__init__(model)
        self.path = path
        self.eta = eta

    def get_train_loss(self, data, device) -> torch.Tensor:
        z, y = data
        z = z.to(device)
        y = y.to(device)
        # mask the image label with rate eta
        mask = torch.rand(z.shape[0]).to(device)
        y[mask < self.eta] = 10.0

        t = torch.rand(z.shape[0], 1, 1, 1).to(device)
        xt = self.path.sample_conditional_path(z, t)

        u_t_ref = self.path.conditional_velocity(xt, z, t)      # shape(bs, c, h, w)
        u_t_theta = self.model(xt, t, y)                              # shape(bs, c, h, w)

        error = torch.einsum("bchw -> b", torch.square(u_t_ref - u_t_theta))    # shape(bs, )
        return torch.mean(error)


Overwriting trainers/fm_trainer.py


## simulators

In [193]:
%%writefile simulators/base.py
from abc import ABC, abstractmethod
import torch

class Simulator(ABC):
    @abstractmethod
    def step(self, xt: torch.Tensor, t: torch.Tensor, dt: torch.Tensor, **kwargs) -> torch.Tensor:
        """
        Takes one simulation step
        Args:
            - xt: state at time t, shape (bs, c, h, w)
            - t: time, shape (bs, 1, 1, 1)
            - dt: time, shape (bs, 1, 1, 1)
        Returns:
            - nxt: state at time t + dt (bs, c, h, w)
        """
        pass

    @torch.no_grad()
    def simulate(self, x: torch.Tensor, ts: torch.Tensor, **kwargs) -> torch.Tensor:
        """
        Simulates using the discretization gives by ts
        Args:
            - x_init: initial state, shape (bs, c, h, w)
            - ts: timesteps, shape (bs, nts, 1, 1, 1)
        Returns:
            - x_final: final state at time ts[-1], shape (bs, c, h, w)
        """
        nts = ts.shape[1]
        for index in range(nts-1):
            t = ts[:, index]
            dt = ts[:, index+1] - t
            x = self.step(x, t, dt, **kwargs)
        return x
        
        
        


Overwriting simulators/base.py


In [194]:
%%writefile simulators/euler.py
from .base import Simulator
from vector_fields import VectorField
import torch

class EulerSimulator(Simulator):
    def __init__(self, vector_field: VectorField):
        self.vector_field = vector_field

    @torch.no_grad()
    def step(self, xt: torch.Tensor, t: torch.Tensor, dt: torch.Tensor, **kwargs) -> torch.Tensor:
        return xt + self.vector_field.velocity(xt, t, **kwargs) * dt


Overwriting simulators/euler.py


## utility functions

In [195]:
%%writefile loss_curve
import matplotlib.pyplot as plt
from typing import List
import os
from pathlib import Path

root_dir = Path("results")

def draw_loss_curve(loss_history: List[float], num_epochs: int, batch_size: int, project_name: str):
    save_path = root_dir / project_name / f"{project_name}_{num_epochs}_epochs_{batch_size}_bs_loss_curve.png"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure()
    plt.plot(loss_history, label=f"{project_name.upper()} Loss")
    plt.yscale('log')
    plt.title(f"{project_name.upper()} Loss Curve with {num_epochs} epochs")
    plt.xlabel("batch")
    plt.ylabel("loss")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


Overwriting loss_curve


In [196]:
%%writefile utils/model_utils.py
import torch
import torch.nn as nn
from pathlib import Path

MiB = 1024 ** 2
root_dir = Path("results")


def model_size_mib(model: nn.Module) -> float:
    size = 0
    for param in model.parameters():
        size += param.nelement() * param.element_size()
    for buf in model.buffers():
        size += buf.nelement() * buf.element_size()
    return size / MiB


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, loss_history,
                    project_name: str, filename: str, is_best: bool = False):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "scaler_state_dict": scaler.state_dict() if scaler else None,
        "epoch": epoch,
        "loss_history": loss_history
    }

    save_path = root_dir / project_name / filename
    save_path.parent.mkdir(parents=True, exist_ok=True)

    torch.save(checkpoint, save_path)
    print(f"[Checkpoint] Saved to {save_path}")

    if is_best:
        best_path = save_path.parent / "best.pth"
        torch.save(checkpoint, best_path)
        print(f"[Checkpoint] Also saved as best model to {best_path}")


def load_checkpoint(model, optimizer, scheduler, scaler,
                    project_name: str, filename: str, device: torch.device = None):
    path = root_dir / project_name / filename
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    epoch = checkpoint["epoch"]
    loss_history = checkpoint.get("loss_history", None)

    if loss_history:
        print(f"[Checkpoint] Loaded from {path}, epoch = {epoch}, loss = {loss_history[-1]:.4f}")
    else:
        print(f"[Checkpoint] Loaded from {path}, epoch = {epoch}, loss = N/A")

    return epoch, loss_history


Overwriting utils/model_utils.py


In [200]:
%%writefile utils/visualization.py
import torch
import math
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from pathlib import Path
from vector_fields import VectorField, CFGVectorField
from simulators import EulerSimulator, HeunSimulator
from paths import ConditionalPath

@torch.no_grad()
def visualize(x: torch.Tensor, save_path: str, dpi: int = 300):
    """
    Visualize a batch of images. Handles normalization automatically.
    Args:
        x: Tensor of shape (B, C, H, W), possibly in range [-1, 1], or unnormalized
        save_path: Where to save the resulting image
    """
    assert x.dim() == 4, f"Expected 4D tensor, got {x.shape}"
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    # --- Normalize input to [0, 1]
    x = x.clone()
    if x.min() < 0:
        x = (x + 1) / 2  # Assume in [-1, 1]
    if x.max() > 1 or x.min() < 0:
        x -= x.min()
        x /= (x.max() + 1e-8)  # Min-max normalization

    # --- Construct image grid
    if x.size(0) == 1:
        img = x[0].permute(1, 2, 0).cpu().numpy()
    else:
        nrow = min(x.size(0), 8)
        grid = make_grid(x, nrow=nrow)
        img = grid.permute(1, 2, 0).cpu().numpy()

    # --- Plot and save
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=dpi)
    plt.close()

@torch.no_grad()
def generate_samples_and_save(
    model: nn.Module,
    path: ConditionalPath,
    device: torch.device,
    save_path: str,
    labels: list,
    guidance_scale: float = 5.0,
    num_steps: int = 100,
    samples_per_label: int = 1,
    simulator_type: str = "euler",
):
    model.eval()
    vector_field = CFGVectorField(model=model, guidance_scale=guidance_scale)

    if simulator_type == "euler":
        simulator = EulerSimulator(vector_field)
    elif simulator_type == "heun":
        simulator = HeunSimulator(vector_field)
    else:
        raise ValueError(f"Unsupported simulator type: {simulator_type}")

    all_images = []

    for label in labels:
        y = torch.full((samples_per_label,), label, dtype=torch.long).to(device)
        x_init = path.noise.sample(samples_per_label)
        x_init = x_init.to(device)
        ts = torch.linspace(0, 1, num_steps).view(1, -1, 1, 1, 1).expand(samples_per_label, -1, -1, -1, -1).to(device)

        x_final = simulator.simulate(x_init, ts, y=y)  # typically in [-1, 1]
        all_images.append(x_final)

    all_images = torch.cat(all_images, dim=0)  # (B, C, H, W)
    visualize(all_images, save_path)


Overwriting utils/visualization.py


In [201]:
%%writefile fm_unet_mnist.py
import torch
from datasets import get_mnist_dataloader
from paths import GaussianConditionalPath, LinearAlpha, LinearBeta
from models import UNetVelocity
from trainers import CFGTrainerFM
from utils import generate_samples_and_save
import matplotlib.pyplot as plt
from vector_fields import CFGVectorField
from simulators import EulerSimulator
from torchvision.utils import make_grid
from matplotlib.axes._axes import Axes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device:{device}")

dataloader = get_mnist_dataloader()
#print("dataloader is done")

path = GaussianConditionalPath(
    p_init_shape = (1, 32, 32),
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

unet = UNetVelocity(
    init_channel = 1,
    channels = [32, 64, 128],
    num_residual_layers = 2,
    t_embed_dim = 64,
    y_embed_dim = 64,
    num_categories = 10
).to(device)

trainer = CFGTrainerFM(
    path = path,
    model = unet,
    eta = 0.1
)

trainer.train(
    num_epochs = 100,
    device = device,
    dataloader = dataloader,
    project_name = "fm_unet_mnist"
)

generate_samples_and_save(
    model = unet,
    path = path,
    device = device,
    save_path = "results/fm_unet_mnist/visualization1.png",
    labels = [i for i in range(1, 11)],
    samples_per_label = 8,
    simulator_type = "heun"
)

## Simple test
samples_per_class = 10
num_timesteps = 100
guidance_scales = [1.0, 3.0, 5.0]

# Graph
fig, axes = plt.subplots(1, len(guidance_scales), figsize=(10 * len(guidance_scales), 10))

for idx, w in enumerate(guidance_scales):
    # Setup ode and simulator
    vector_field = CFGVectorField(unet, guidance_scale=w)
    simulator = EulerSimulator(vector_field)

    # Sample initial conditions
    y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int64).repeat_interleave(samples_per_class).to(device)
    num_samples = y.shape[0]
    x0 = path.noise.sample(num_samples) # (num_samples, 1, 32, 32)

    # Simulate
    ts = torch.linspace(0,1,num_timesteps).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1).to(device)
    x1 = simulator.simulate(x0, ts, y=y)

    # Plot
    grid = make_grid(x1, nrow=samples_per_class, normalize=True, value_range=(-1,1))
    axes[idx].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    axes[idx].axis("off")
    axes[idx].set_title(f"Guidance: $w={w:.1f}$", fontsize=25)
plt.savefig("results/fm_unet_mnist/visualization2.png", dpi=300)


Overwriting fm_unet_mnist.py
